# Baseline models

**Objective:** reproducible reference scores before any tuning or heavy
feature engineering.

Questions:

1. How much signal do demographic, calendar and climate variables carry on their own?
2. Does a linear model already work?
3. Does a tree-based model clearly beat it?
4. What is the first reliable competition score?

All scores use the grouped validation chosen in `Validation.ipynb`.

## Imports

In [1]:
import sys
from pathlib import Path

# Make the project root importable (the notebook runs from notebooks/)
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.config import LOCATION_COL, TARGET, TRAIN_MERGED
from src.features import get_feature_groups, prepare_features
from src.models import make_builder
from src.validation import cross_validate, evaluate_oof, make_folds

pd.set_option("display.width", 150)

## Load data and build folds

In [2]:
train = pd.read_csv(TRAIN_MERGED)
y = train[TARGET]

X = prepare_features(train, feature_set="base", drop_location_features=True)
folds = make_folds(y, "group", groups=train[LOCATION_COL])

groups = get_feature_groups(X)
for name, cols in groups.items():
    print(f"{name:12s} ({len(cols):2d}):", cols)

demographic  ( 3): ['zone', 'gender', 'age']
calendar     ( 3): ['death_year', 'death_month', 'death_dayofyear']
location     ( 0): []
climate      (18): ['avg_temperature', 'max_temperature', 'min_temperature', 'precipitation', 'hot_days_30d', 'max_daily_rain_30d', 'ndvi_30d', 'ndvi_90d', 'rain_days_30d', 'rain_sum_30d', 'rain_sum_7d', 'rain_sum_90d', 'tavg_30d', 'tavg_7d', 'tavg_90d', 'temp_range_mean_30d', 'tmax_30d', 'tmin_30d']


## Signal by feature group

Each model is trained on one group of variables at a time, then on all of them.

In [3]:
subsets = {
    "demographic": groups["demographic"],
    "calendar": groups["calendar"],
    "climate": groups["climate"],
    "all": list(X.columns),
}

rows = []
for model_name in ["logreg", "catboost"]:
    for subset_name, cols in subsets.items():
        X_sub = X[cols]
        result = cross_validate(make_builder(model_name, X_sub), X_sub, y, folds)
        m = evaluate_oof(y, result)
        rows.append({
            "model": model_name,
            "features": subset_name,
            "threshold": m["threshold"],
            "F1": round(m["F1"], 4),
            "ROC_AUC": round(m["ROC_AUC"], 4),
            "score": round(m["score"], 4),
            "fold_auc_std": round(m["fold_auc_std"], 4),
        })

baseline_table = pd.DataFrame(rows)
baseline_table.sort_values("score", ascending=False)

,model,features,threshold,F1,ROC_AUC,score,fold_auc_std
3,logreg,all,0.30,0.8199,0.8019,0.8127,0.0165
7,catboost,all,0.36,0.8159,0.8059,0.8119,0.0079
4,catboost,demographic,0.49,0.8096,0.7700,0.7938,0.0186
0,logreg,demographic,0.28,0.8163,0.7465,0.7884,0.0239
1,logreg,calendar,0.05,0.7884,0.6047,0.7149,0.0159
5,catboost,calendar,0.05,0.7884,0.5917,0.7097,0.0133
6,catboost,climate,0.05,0.7884,0.5659,0.6994,0.0130
2,logreg,climate,0.27,0.7887,0.5560,0.6956,0.0296


## Reading the table

- Compare every score to the trivial predictor (see `Validation.ipynb`):
  F1 is already high because the positive class is the majority.
  **ROC-AUC is the more informative column** for judging real signal.
- A feature group with ROC-AUC close to 0.5 carries little information.
- If CatBoost does not clearly beat Logistic Regression on `all`, the
  relationship is mostly simple and gains must come from better features.